# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [17]:
import pandas as pd
import numpy as np
import seaborn  as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest


In [18]:
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')# tu código aquí
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')# tu código aquí
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')# tu código aquí

In [19]:

display(orders.head())
print(orders.info())
print(orders.describe())


    

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
None
           cantidad  precio_unitario  monto_descuento   monto_total
count  25050.000000     25050.000000     25050.000000  2.510000e+04
mean       7

In [20]:
display(catalog.head(7))
print(catalog.info())
print(catalog.describe())


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
None
       costo_unitario
count        7.000000
mean       102.252857
std        111.011563
min         10.120000
25%         16.905000
50%         25.210000
75%        182.975000
max        280.680000


In [21]:
display(marketing.head())
print(marketing.info())
print(marketing.describe())


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None
            gasto
count  1620.00000
mean   1772.74292
std     734.43294
min     501.11000
25%    1128.03000
50%    1782.42500
75%    2420.68500
max    2999.36000


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.


In [22]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'],errors="coerce")
marketing['fecha'] = pd.to_datetime(marketing['fecha'],errors="coerce")
display(orders.head())
display(marketing.head())


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [23]:
print("MIN",orders[['cantidad','monto_total']].min())
print("MAX",orders[['cantidad','monto_total']].max())

MIN cantidad        -2.00
monto_total   -492.65
dtype: float64
MAX cantidad         20000.0
monto_total    8840200.0
dtype: float64


In [24]:
print(orders[orders['cantidad'] <= 0].shape[0])
print(orders[orders['cantidad'] > 2].shape[0])

print(orders['cantidad'].value_counts())



4
10
 2.0        12642
 1.0        12394
 10000.0        6
 20000.0        4
-1.0            3
-2.0            1
Name: cantidad, dtype: int64


In [25]:
cantidades_anomalas = orders[orders['cantidad'].isin([-1, -2, 10000, 20000])]

cantidades_anomalas[
    ['id_pedido', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
]

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,-2.0,101.31,10.0,-192.62
267,order_267,-1.0,43.50,5.0,-38.50
268,order_268,-1.0,497.65,5.0,-492.65
269,order_269,-1.0,423.53,0.0,-423.53
3521,order_3521,10000.0,43.14,0.0,431400.00
3522,order_3522,10000.0,280.55,0.0,2805500.00
3586,order_3586,10000.0,490.35,0.0,4903500.00
3643,order_3643,10000.0,238.15,0.0,2381500.00
3656,order_3656,20000.0,297.66,0.0,5953200.00
3668,order_3668,20000.0,348.31,0.0,6966200.00


In [26]:
orders['cantidad'] = orders['cantidad'].replace({-1:1,-2:2,10000:1,20000:2})
orders.describe()


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,1.504870,259.305549,4.500798,2.072680e+03
std,0.499986,138.726461,5.223010,9.894995e+04
min,1.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,2.000000,499.960000,15.000000,8.840200e+06


In [27]:
orders.loc[
    cantidades_anomalas.index,
    ['id_pedido', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
]

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,2.0,101.31,10.0,-192.62
267,order_267,1.0,43.50,5.0,-38.50
268,order_268,1.0,497.65,5.0,-492.65
269,order_269,1.0,423.53,0.0,-423.53
3521,order_3521,1.0,43.14,0.0,431400.00
3522,order_3522,1.0,280.55,0.0,2805500.00
3586,order_3586,1.0,490.35,0.0,4903500.00
3643,order_3643,1.0,238.15,0.0,2381500.00
3656,order_3656,2.0,297.66,0.0,5953200.00
3668,order_3668,2.0,348.31,0.0,6966200.00


In [28]:
orders.loc[cantidades_anomalas.index, 'monto_total'] = (
    orders.loc[cantidades_anomalas.index, 'cantidad']
    * orders.loc[cantidades_anomalas.index, 'precio_unitario']
    - orders.loc[cantidades_anomalas.index, 'monto_descuento']
).round(2)

In [29]:
orders.loc[
    cantidades_anomalas.index,
    ['id_pedido', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
]

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,2.0,101.31,10.0,192.62
267,order_267,1.0,43.50,5.0,38.50
268,order_268,1.0,497.65,5.0,492.65
269,order_269,1.0,423.53,0.0,423.53
3521,order_3521,1.0,43.14,0.0,43.14
3522,order_3522,1.0,280.55,0.0,280.55
3586,order_3586,1.0,490.35,0.0,490.35
3643,order_3643,1.0,238.15,0.0,238.15
3656,order_3656,2.0,297.66,0.0,595.32
3668,order_3668,2.0,348.31,0.0,696.62


In [30]:
orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,25100.000000
mean,1.504870,259.305549,4.500798,385.899866
std,0.499986,138.726461,5.223010,255.742755
min,1.000000,20.030000,0.000000,5.240000
25%,1.000000,138.377500,0.000000,180.555000
50%,2.000000,258.715000,0.000000,341.605000
75%,2.000000,380.332500,10.000000,517.922500
max,2.000000,499.960000,15.000000,999.890000


In [32]:
print(orders[orders['monto_total'] < 0].shape[0])
print(orders[orders['precio_unitario'] < 0].shape[0])
print(orders[orders['monto_descuento'] < 0].shape[0])






0
0
0


In [33]:
orders[orders['monto_total'] < 0][['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']]

,cantidad,precio_unitario,monto_descuento,monto_total


In [36]:
print(orders['id_pedido'].duplicated().sum())
print(catalog['nombre_producto'].duplicated().sum())
print(marketing.duplicated(subset=['id_campaña', 'pais', 'canal', 'fecha', 'gasto']).sum())
orders[orders['id_pedido'].duplicated(keep=False)].sort_values('id_pedido')

0
0
0


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total


In [34]:
orders = orders.drop_duplicates() 
print(orders['id_pedido'].duplicated().sum())

0


In [35]:
display(orders.isna().sum())

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

In [37]:
display(marketing.isna().sum())
marketing[marketing['canal'].isna()]

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
...,...,...,...,...,...
194,2025-01-22,Colombia,social_Colombia,NaN,772.46
195,2025-01-22,Argentina,organic_Argentina,NaN,1562.61
196,2025-01-22,Argentina,paid_search_Argentina,NaN,644.34
197,2025-01-22,Argentina,social_Argentina,NaN,2544.66


In [38]:
marketing['canal'] = marketing['canal'].fillna("Desconocido")
display(marketing.isna().sum())


fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

In [39]:
orders.loc[orders['monto_descuento'].isna(),'monto_descuento'] = 0

indices_a_eliminar = orders[orders['cantidad'].isna() & orders['precio_unitario'].isna()].index
orders = orders.drop(indices_a_eliminar)



In [40]:
orders.loc[orders['cantidad'].isna(),'cantidad'] = ((orders["monto_total"] + orders["monto_descuento"])/ orders["precio_unitario"])
orders.loc[orders['precio_unitario'].isna(),'precio_unitario'] = ((orders["monto_total"] + orders["monto_descuento"])/ orders["cantidad"])
display(orders.isna().sum())

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64

In [42]:
orders[['pais', 'dispositivo', 'nombre_producto', 'categoria_producto','fuente_referencia']]= orders[['pais', 'dispositivo', 'nombre_producto', 'categoria_producto','fuente_referencia']].fillna("Desconocido")
display(orders.isna().sum())

id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64

In [43]:
orders['cantidad'] = orders['cantidad'].astype(int)
orders.head()


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1,336.28,0.0,336.28


In [44]:

display(orders['pais'].unique(),
orders['dispositivo'].unique(),
orders['fuente_referencia'].unique(),
orders['nombre_producto'].unique(),
orders['categoria_producto'].unique())



array(['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia',
       'Desconocido', 'argentina'], dtype=object)

array(['desktop', 'mobile', 'Desconocido'], dtype=object)

array(['organic', 'paid_search', 'social', 'Desconocido'], dtype=object)

array(['Jacket-Winter-M', 'Tablet-Standard-64GB', 'Blender-XL-Red',
       'Laptop-Gaming-16GB', 'Sneakers-Urban-42', 'Phone-Pro-128GB',
       'Vacuum-Pro-Black', 'Desconocido'], dtype=object)

array(['Moda', 'Electronica', 'Hogar', 'Desconocido'], dtype=object)

In [45]:

orders['pais'] = orders['pais'].str.title()
orders['dispositivo'] = orders['dispositivo'].str.title()
orders['fuente_referencia'] = orders['fuente_referencia'].str.title()

orders['pais'] = orders['pais'].str.strip()
orders['dispositivo'] = orders['dispositivo'].str.strip()
orders['fuente_referencia'] = orders['fuente_referencia'].str.strip()
orders['nombre_producto'] = orders['nombre_producto'].str.strip()
orders['categoria_producto'] = orders['categoria_producto'].str.strip()



display(orders['pais'].unique(),
orders['dispositivo'].unique(),
orders['fuente_referencia'].unique(),
 orders['nombre_producto'].unique(),
orders['categoria_producto'].unique())

array(['Argentina', 'Mexico', 'Colombia', 'Desconocido'], dtype=object)

array(['Desktop', 'Mobile', 'Desconocido'], dtype=object)

array(['Organic', 'Paid_Search', 'Social', 'Desconocido'], dtype=object)

array(['Jacket-Winter-M', 'Tablet-Standard-64GB', 'Blender-XL-Red',
       'Laptop-Gaming-16GB', 'Sneakers-Urban-42', 'Phone-Pro-128GB',
       'Vacuum-Pro-Black', 'Desconocido'], dtype=object)

array(['Moda', 'Electronica', 'Hogar', 'Desconocido'], dtype=object)

In [46]:


display(marketing['pais'].unique(),
marketing['canal'].unique(),
marketing['id_campaña'].unique())





array(['Mexico', 'Colombia', 'Argentina'], dtype=object)

array(['organic', 'paid_search', 'social', 'Desconocido'], dtype=object)

array(['organic_Mexico', 'paid_search_Mexico', 'social_Mexico',
       'organic_Colombia', 'paid_search_Colombia', 'social_Colombia',
       'organic_Argentina', 'paid_search_Argentina', 'social_Argentina'],
      dtype=object)

In [47]:
marketing['canal'] = marketing['canal'].str.title()
marketing['id_campaña'] = marketing['id_campaña'].str.title()
marketing['pais'] = marketing['pais'].str.strip()
marketing['id_campaña'] = marketing['id_campaña'].str.strip()
marketing['canal'] = marketing['canal'].str.strip()

display(marketing['pais'].unique(),
marketing['canal'].unique(),
marketing['id_campaña'].unique())


array(['Mexico', 'Colombia', 'Argentina'], dtype=object)

array(['Organic', 'Paid_Search', 'Social', 'Desconocido'], dtype=object)

array(['Organic_Mexico', 'Paid_Search_Mexico', 'Social_Mexico',
       'Organic_Colombia', 'Paid_Search_Colombia', 'Social_Colombia',
       'Organic_Argentina', 'Paid_Search_Argentina', 'Social_Argentina'],
      dtype=object)

In [48]:
display(catalog['nombre_producto'].unique(),
catalog['categoria_producto'].unique(),
catalog['proveedor'].unique())


array(['Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Tablet-Standard-64GB',
       'Blender-XL-Red', 'Vacuum-Pro-Black', 'Sneakers-Urban-42',
       'Jacket-Winter-M'], dtype=object)

array(['Electrónica', 'Hogar', 'Moda'], dtype=object)

array(['Fuller, Pena and Myers', 'King Ltd', 'Bowers LLC', 'Long-Reid',
       'Rivera, Carr and Finley', 'Greene-Smith', 'Mcmillan-Rhodes'],
      dtype=object)

In [49]:
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip()
catalog['proveedor'] = catalog['proveedor'].str.strip()

catalog["categoria_producto"]= catalog['categoria_producto'].replace({"Electrónica" : "Electronica"})

display(catalog['nombre_producto'].unique(),
catalog['categoria_producto'].unique(),
catalog['proveedor'].unique())

array(['Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Tablet-Standard-64GB',
       'Blender-XL-Red', 'Vacuum-Pro-Black', 'Sneakers-Urban-42',
       'Jacket-Winter-M'], dtype=object)

array(['Electronica', 'Hogar', 'Moda'], dtype=object)

array(['Fuller, Pena and Myers', 'King Ltd', 'Bowers LLC', 'Long-Reid',
       'Rivera, Carr and Finley', 'Greene-Smith', 'Mcmillan-Rhodes'],
      dtype=object)

In [50]:
display(orders.head(),catalog.head(),marketing.head())

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,Desktop,Social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1,336.28,0.0,336.28


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electronica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electronica,10.12,King Ltd
2,Tablet-Standard-64GB,Electronica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,Organic_Mexico,Organic,2446.25
1,2025-01-01,Mexico,Paid_Search_Mexico,Paid_Search,2704.34
2,2025-01-01,Mexico,Social_Mexico,Social,2045.01
3,2025-01-01,Colombia,Organic_Colombia,Organic,2597.21
4,2025-01-01,Colombia,Paid_Search_Colombia,Paid_Search,1771.40


In [52]:

orders.to_csv('orders_clean.csv', index=False, sep= ';', decimal = ',')
catalog.to_csv('catalog_clean.csv', index=False,sep= ';', decimal = ',')
marketing.to_csv('marketing_clean.csv', index=False,sep= ';', decimal = ',')

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [53]:
revenue = orders['monto_total'].sum()
print("Revenue", revenue)

Revenue 9627663.33


In [55]:
order_with_cost = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how ="left")

In [56]:
print("Filas en orders:", len(orders))
print("Filas después del merge:", len(order_with_cost))

Filas en orders: 24950
Filas después del merge: 24950


In [57]:
costo = order_with_cost['cantidad'] * order_with_cost['costo_unitario']
costo_total = costo.sum()
print (costo_total)

3832849.13


In [58]:
inversion_marketing = marketing['gasto'].sum()
print("inversion marketing", inversion_marketing)

inversion marketing 2871843.53


In [59]:
ganancia_bruta = revenue - costo_total
print("Ganancia Bruta", ganancia_bruta)

Ganancia Bruta 5794814.2


In [60]:

ganancia_neta = ganancia_bruta - inversion_marketing
print("Ganancia Neta", ganancia_neta)

Ganancia Neta 2922970.6700000004


In [61]:
margen_ganancia = (ganancia_neta / revenue ) * 100
print ("Margen de ganancia", margen_ganancia)

Margen de ganancia 30.360125503059116


In [62]:
ticket_promedio = orders['monto_total'].mean()
print("Ticket promedio", ticket_promedio)

Ticket promedio 385.8782897795591


In [63]:
cantidad_promedio = orders['cantidad'].mean()
print("Cantidad promedio de productos por orden", cantidad_promedio)


Cantidad promedio de productos por orden 1.5048496993987976


In [64]:
productos_mas_vendidos = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print(productos_mas_vendidos)
producto_mas_vendido = productos_mas_vendidos.index[0]
print("Producto mas Vendido : ", producto_mas_vendido)

nombre_producto
Vacuum-Pro-Black        6284
Blender-XL-Red          6279
Jacket-Winter-M         6256
Sneakers-Urban-42       6172
Laptop-Gaming-16GB      4212
Tablet-Standard-64GB    4153
Phone-Pro-128GB         4145
Desconocido               45
Name: cantidad, dtype: int64
Producto mas Vendido :  Vacuum-Pro-Black


In [65]:
gastos_marketing_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print("Gasto de marketing segun canal", gastos_marketing_canal)
canal_mayor_gasto_marketing = gastos_marketing_canal.index[0]
print ("Canal con mayor gasto en marketing: ",canal_mayor_gasto_marketing)

Gasto de marketing segun canal canal
Social         918043.21
Organic        913533.01
Paid_Search    863088.21
Desconocido    177179.10
Name: gasto, dtype: float64
Canal con mayor gasto en marketing:  Social


---

funnel de conversión

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [74]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Credenciales de conexión: configurar como variables de entorno locales.
# Nunca publicar usuarios, contraseñas ni direcciones privadas en GitHub.
required = ["RAPPIPLUS_DB_USER", "RAPPIPLUS_DB_PASSWORD", "RAPPIPLUS_DB_HOST", "RAPPIPLUS_DB_NAME"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise EnvironmentError("Configura las variables de entorno de la base de datos para ejecutar las consultas SQL.")

connection_url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["RAPPIPLUS_DB_USER"],
    password=os.environ["RAPPIPLUS_DB_PASSWORD"],
    host=os.environ["RAPPIPLUS_DB_HOST"],
    port=int(os.getenv("RAPPIPLUS_DB_PORT", "5432")),
    database=os.environ["RAPPIPLUS_DB_NAME"],
)
engine = create_engine(connection_url, connect_args={"sslmode": "require"})


In [75]:

query_events = '''
SELECT * 
FROM events
LIMIT 10 ;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [76]:
query_eventos = '''
SELECT
    id_usuario,
    nombre_evento,
    timestamp_evento
FROM events
ORDER BY id_usuario, timestamp_evento
LIMIT 20;
'''

eventos_ordenados = pd.read_sql(query_eventos, con=engine)
eventos_ordenados

,id_usuario,nombre_evento,timestamp_evento
0,user_0,add_to_cart,2025-01-22
1,user_0,add_to_cart,2025-01-23
2,user_0,first_visit,2025-01-24
3,user_0,add_payment_info,2025-03-17
4,user_0,first_visit,2025-04-19
5,user_0,select_item,2025-04-26
6,user_0,select_item,2025-04-29
7,user_0,select_item,2025-05-05
8,user_0,select_item,2025-05-12
9,user_0,select_item,2025-06-10


In [77]:

etapas_funnel = '''

SELECT DISTINCT nombre_evento
FROM events;
'''



totals = pd.read_sql(etapas_funnel, con=engine)
totals

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [78]:
query_totals = '''
WITH first_visit AS (
    SELECT
        id_usuario,
        MIN(timestamp_evento) AS fecha_first_visit
    FROM events
    WHERE nombre_evento = 'first_visit'
    GROUP BY id_usuario
),

select_item AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS fecha_select_item
    FROM events e
    INNER JOIN first_visit f
        ON e.id_usuario = f.id_usuario
    WHERE e.nombre_evento = 'select_item'
      AND e.timestamp_evento >= f.fecha_first_visit
    GROUP BY e.id_usuario
),

add_to_cart AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS fecha_add_to_cart
    FROM events e
    INNER JOIN select_item s
        ON e.id_usuario = s.id_usuario
    WHERE e.nombre_evento = 'add_to_cart'
      AND e.timestamp_evento >= s.fecha_select_item
    GROUP BY e.id_usuario
),

begin_checkout AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS fecha_begin_checkout
    FROM events e
    INNER JOIN add_to_cart a
        ON e.id_usuario = a.id_usuario
    WHERE e.nombre_evento = 'begin_checkout'
      AND e.timestamp_evento >= a.fecha_add_to_cart
    GROUP BY e.id_usuario
),

add_payment_info AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS fecha_add_payment_info
    FROM events e
    INNER JOIN begin_checkout b
        ON e.id_usuario = b.id_usuario
    WHERE e.nombre_evento = 'add_payment_info'
      AND e.timestamp_evento >= b.fecha_begin_checkout
    GROUP BY e.id_usuario
),

purchase AS (
    SELECT
        e.id_usuario,
        MIN(e.timestamp_evento) AS fecha_purchase
    FROM events e
    INNER JOIN add_payment_info p
        ON e.id_usuario = p.id_usuario
    WHERE e.nombre_evento = 'purchase'
      AND e.timestamp_evento >= p.fecha_add_payment_info
    GROUP BY e.id_usuario
)

SELECT
    (SELECT COUNT(*) FROM first_visit) AS first_visit_users,
    (SELECT COUNT(*) FROM select_item) AS select_item_users,
    (SELECT COUNT(*) FROM add_to_cart) AS add_to_cart_users,
    (SELECT COUNT(*) FROM begin_checkout) AS begin_checkout_users,
    (SELECT COUNT(*) FROM add_payment_info) AS add_payment_info_users,
    (SELECT COUNT(*) FROM purchase) AS purchase_users;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,first_visit_users,select_item_users,add_to_cart_users,begin_checkout_users,add_payment_info_users,purchase_users
0,7796,6728,4983,2663,919,258


In [79]:
conversion = totals.copy()

conversion['tasa_conversion_select_item'] = (
    conversion['select_item_users'] / conversion['first_visit_users'] * 100
)

conversion['tasa_conversion_add_to_cart'] = (
    conversion['add_to_cart_users'] / conversion['select_item_users'] * 100
)

conversion['tasa_conversion_begin_checkout'] = (
    conversion['begin_checkout_users'] / conversion['add_to_cart_users'] * 100
)

conversion['tasa_conversion_add_payment_info'] = (
    conversion['add_payment_info_users'] / conversion['begin_checkout_users'] * 100
)

conversion['tasa_conversion_purchase'] = (
    conversion['purchase_users'] / conversion['add_payment_info_userconversion.round(2)

,first_visit_users,select_item_users,add_to_cart_users,begin_checkout_users,add_payment_info_users,purchase_users,tasa_conversion_select_item,tasa_conversion_add_to_cart,tasa_conversion_begin_checkout,tasa_conversion_add_payment_info,tasa_conversion_purchase
0,7796,6728,4983,2663,919,258,86.3,74.06,53.44,34.51,28.07


In [80]:
tasas = conversion.filter(like='tasa_conversion_')

menor_conversion = tasas.iloc[0].idxmin()
valor_conversion = tasas.iloc[0].min()

print("Etapa con menor conversión:", menor_conversion)
print(f"Tasa de conversión: {valor_conversion:.2f}%")


Etapa con menor conversión: tasa_conversion_purchase
Tasa de conversión: 28.07%


---

## 🔹Rretención por cohortes


In [81]:
query_users = '''


SELECT id_usuario, país, dispositivo, tipo_plan,
CAST(fecha_registro AS DATE)
FROM users
;

'''
users = pd.read_sql(query_users, con=engine)
users.head()

,id_usuario,país,dispositivo,tipo_plan,fecha_registro
0,user_0,Mexico,mobile,free,2025-01-29
1,user_1,Mexico,mobile,free,2025-01-07
2,user_2,Argentina,mobile,free,2025-03-12
3,user_3,Mexico,desktop,free,2025-03-04
4,user_4,Argentina,desktop,free,2025-02-27


In [84]:
query_user_activity = '''
SELECT id_usuario, dias_despues_registro, activo, 
CAST (fecha_actividad AS DATE)
FROM user_activity
;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,dias_despues_registro,activo,fecha_actividad
0,user_0,7,0,2025-02-05
1,user_0,14,1,2025-02-12
2,user_0,21,1,2025-02-19


In [85]:
query_cohort_retention_final = '''

WITH cohortes AS (
    SELECT
        id_usuario,
        TO_CHAR(
            DATE_TRUNC('month', CAST(fecha_registro AS DATE)),
            'YYYY-MM'
        ) AS cohort_mes
    FROM users
),

actividad AS (
    SELECT
        a.id_usuario,
        a.dias_despues_registro,
        a.activo,
        c.cohort_mes
    FROM user_activity a
    INNER JOIN cohortes c
        ON a.id_usuario = c.id_usuario
),

retencion AS (
    SELECT
        cohort_mes,

        COUNT(DISTINCT CASE
            WHEN dias_despues_registro >= 7
                 AND activo = 1
            THEN id_usuario
        END) AS retention_w1,

        COUNT(DISTINCT CASE
            WHEN dias_despues_registro >= 14
                 AND activo = 1
            THEN id_usuario
        END) AS retention_w2,

        COUNT(DISTINCT CASE
            WHEN dias_despues_registro >= 21
                 AND activo = 1
            THEN id_usuario
        END) AS retention_w3

    FROM actividad
    GROUP BY cohort_mes
),

tamano_cohorte AS (
    SELECT
        TO_CHAR(
            DATE_TRUNC('month', CAST(fecha_registro AS DATE)),
            'YYYY-MM'
        ) AS cohort_mes,
        COUNT(DISTINCT id_usuario) AS cohort_size
    FROM users
    GROUP BY cohort_mes
)

SELECT
    r.cohort_mes,
    t.cohort_size,

    r.retention_w1,
    r.retention_w2,
    r.retention_w3,

    ROUND(100.0 * r.retention_w1 / t.cohort_size, 2)
        AS retention_w1_pct,

    ROUND(100.0 * r.retention_w2 / t.cohort_size, 2)
        AS retention_w2_pct,

    ROUND(100.0 * r.retention_w3 / t.cohort_size, 2)
        AS retention_w3_pct

FROM retencion r
INNER JOIN tamano_cohorte t
    ON r.cohort_mes = t.cohort_mes

ORDER BY r.cohort_mes;

'''

cohorte_final = pd.read_sql(
    query_cohort_retention_final,
    con=engine
)

cohorte_final

,cohort_mes,cohort_size,retention_w1,retention_w2,retention_w3,retention_w1_pct,retention_w2_pct,retention_w3_pct
0,2025-01,1627,1381,1253,1027,84.88,77.01,63.12
1,2025-02,1444,1255,1154,940,86.91,79.92,65.10
2,2025-03,1636,1428,1306,1060,87.29,79.83,64.79
3,2025-04,1606,1394,1261,1022,86.80,78.52,63.64
4,2025-05,1687,1446,1321,1088,85.71,78.30,64.49


### Interpretación de retención por cohortes

La retención acumulada presenta un comportamiento consistente entre las cohortes analizadas. En la semana 1, entre aproximadamente el 85 % y 87 % de los usuarios continúa activo. En la semana 2, la retención disminuye a valores cercanos al 77 %–80 %, mientras que en la semana 3 se sitúa entre aproximadamente el 63 % y 65 %.

La cohorte de febrero presenta la mayor retención en la semana 3 (65,10 %), seguida por marzo (64,79 %) y mayo (64,49 %).

Los resultados muestran que la mayor pérdida de usuarios ocurre progresivamente durante las primeras tres semanas posteriores al registro. Esto sugiere una oportunidad para reforzar las estrategias de activación y fidelización durante este período, especialmente antes de la tercera semana.

---



## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)



🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.



---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** Las tasas de conversión de control y tratamiento son iguales.
   - **H₁ (Hipótesis alternativa):** Las tasas de conversión de control y tratamiento son diferentes.
   
**Test estadístico:** z-test  
**Nivel de significancia alpha:** 0.05

In [166]:

experimento_checkout= pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
display(experimento_checkout.head())
experimento_checkout.info()




,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [167]:
print(experimento_checkout['variante'].describe())
print("\nConteo variantes:")
print(experimento_checkout['variante'].value_counts())

count           10000
unique              2
top       tratamiento
freq             5035
Name: variante, dtype: object

Conteo variantes:
tratamiento    5035
control        4965
Name: variante, dtype: int64


In [168]:

conversiones = experimento_checkout.groupby('variante')['convirtio'].sum()
totales = experimento_checkout.groupby('variante')['convirtio'].count()
exitos = [conversiones['control'], conversiones['tratamiento']]
observaciones = [totales['control'], totales['tratamiento']]
z_stat, p_value = proportions_ztest(exitos, observaciones)
print ("z_stat :",z_stat ,"p_value:",p_value)
print("Usuarios convertidos por variante:",conversiones)
print("Total de usuarios por variante:",totales)

z_stat : -0.8132782986429474 p_value: 0.41605851639119995
Usuarios convertidos por variante: variante
control        779
tratamiento    820
Name: convirtio, dtype: int64
Total de usuarios por variante: variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64


In [169]:
tasa_control = [conversiones['control'] / totales['control']]
tasa_tratamiento = [conversiones['tratamiento'] / totales['tratamiento']]

print(f"Tasa de conversión grupo control {tasa_control}")
print(f"Tasa de conversión grupo tratamiento {tasa_tratamiento}")

Tasa de conversión grupo control [0.1568982880161128]
Tasa de conversión grupo tratamiento [0.1628599801390268]


Los datos crudos demuestras que la variante "tratamiento" tiene una tasa de conversion numéricamente mayor que la variante "control".Sin embargo el valor de p-value es mayor que el nivel de significancia, por lo que no se rechaza la hipotesis nula planteada, la pequeña diferencia observada no mantiene evidencia suficiente para afirmar que el cambio de UI realmente mejora la conversión así que implementar estrategías sobre el cambio de UI no sería una acción recomendable. Por otra parte se puede reevaluar el impacto con una muestra mayor como también aumentar el periodo del experimento, además puede ser materia de análisis si el efecto del experimento impacta de igual manera en todos los tipos de usuarios.

---